In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv("10-diamonds.csv")

In [3]:
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [4]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [5]:
df.head(3)

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31


In [6]:
df = df.drop(df[df['x'] == 0].index)
df = df.drop(df[df['y'] == 0].index)
df = df.drop(df[df['z'] == 0].index)

In [7]:
df = df[(df['depth'] < 75 ) & (df['depth'] > 45)]
df = df[(df['table'] < 80 ) & (df['table'] > 40)]
df = df[(df['y'] < 30 )]
df = df[(df['z'] < 30 ) & (df['z'] > 2)]

In [8]:
X = df.drop('price', axis=1)
y = df['price']

In [9]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.25, random_state=42)

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
#label_encoder = LabelEncoder()

#for col in ["cut", "color", "clarity"]:
#    X_train[col] = label_encoder.fit_transform(X_train[col])
#    X_test[col] = label_encoder.transform(X_test[col])

# daha önce sütunları LabelEncoder ile bu şekilde encode etmiştik ve çalışıyordu
# ancak encoder’ları kaydetmek istiyorsak, onları ayrı ayrı tanımlamamız gerekiyordu
# bu yüzden yeni bir encoding yöntemi kullanacağım çünkü kullanıcı encode edilmiş şekilde değer girmeyecek. Scale edilmemiş hatta kategorik 
# değerler bile girebilecek ve biz buna rağmen modeli çalıştıracağımız için farklı bir yöntem kullanacağız.

In [13]:
# Her değişkeni farklı encoderlar ile encode ediyoruz mantık bu.

encoders = {}

for col in ['cut', 'color', 'clarity']:
    encoders[col] = LabelEncoder()
    X_train[col] = encoders[col].fit_transform(X_train[col])
    X_test[col] = encoders[col].transform(X_test[col])

In [14]:
X_train.head()

,carat,cut,color,clarity,depth,table,x,y,z
3981,1.00,0,4,2,65.5,57.0,6.26,6.21,4.08
7096,1.00,0,4,5,66.8,63.0,6.19,6.08,4.10
33892,0.30,2,1,5,62.6,54.0,4.31,4.28,2.69
36969,0.34,2,0,4,60.7,57.0,4.55,4.51,2.75
18486,1.00,4,3,7,62.3,58.0,6.39,6.45,4.00


In [15]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [16]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
from sklearn.svm import SVR

In [18]:
svr = SVR(C=1000, gamma=0.1, kernel="rbf")

In [19]:
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

In [20]:
from sklearn.metrics import r2_score

In [21]:
score = r2_score(y_test,y_pred)
print("R2 SCORE: ", score)

R2 SCORE:  0.9424255496384273


In [22]:
encoders

{'cut': LabelEncoder(), 'color': LabelEncoder(), 'clarity': LabelEncoder()}

In [23]:
scaler

,copy,True
,with_mean,True
,with_std,True


In [24]:
svr

,kernel,'rbf'
,degree,3
,gamma,0.1
,coef0,0.0
,tol,0.001
,C,1000
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [25]:
# Görüldüğü gibi encoder , scaler ve modelimiz kayıtlı. Her seferinde en baştan oluşturmak yerine kaydettik ve bunu kullanarak hesaplamalar yapacağız

In [26]:
import pickle

In [27]:
with open("30-diamond_model_complete.pkl", "wb") as f: # # wb = wright binary
    pickle.dump(
        {
            'model': svr,
            "encoders": encoders,
            "scaler": scaler
            
        }
    ,f)

In [28]:
# nasıl çalıştığını göstermek için bunu da kaydedip gözle görelim.
pd.DataFrame(X_test_scaled).to_csv("30-testdatascaled.csv", index=False)